# Two-tower inference on ESCI

This notebook loads a trained two-tower model and FAISS index and runs retrieval for ad-hoc queries.

In [ ]:
import sys
from pathlib import Path

# Project root
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.constants import DATA_DIR, DEFAULT_MODEL_NAME

DATA_DIR, DEFAULT_MODEL_NAME

In [ ]:
import logging

import torch

from src.models.two_tower import TwoTowerEncoder
from src.retrieval.build_index import load_index_and_meta
from src.retrieval.query import search

logging.basicConfig(level=logging.INFO, format="%(message)s")

In [ ]:
# Load trained model and FAISS index

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = DATA_DIR / "model.pt"
index_path = DATA_DIR / "product.index"
meta_path = DATA_DIR / "product_meta.parquet"

ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
model_name = ckpt.get("model_name", DEFAULT_MODEL_NAME)
model = TwoTowerEncoder(model_name=model_name, shared=False, normalize=True).to(DEVICE)
model.load_state_dict(ckpt["model_state"], strict=True)
model.eval()

index, meta_df = load_index_and_meta(index_path, meta_path)
len(meta_df)

In [ ]:
# Helper to run a single query through the retriever

def retrieve(query: str, top_k: int = 10):
    results = search(
        query=query,
        model=model,
        index=index,
        meta_df=meta_df,
        top_k=top_k,
        device=DEVICE,
    )
    for rank, (pid, score) in enumerate(results, start=1):
        text = meta_df.loc[meta_df["product_id"] == pid, "product_text"].iloc[0]
        print(f"{rank:2d}. product_id={pid}  score={score:.4f}")
        print(f"    {text[:240]}\n")

In [ ]:
# Try a few example queries

queries = [
    "organic whole milk",
    "wireless headphones",
    "dog treats grain free",
]

for q in queries:
    print(f"Query: {q}")
    retrieve(q, top_k=5)
    print("=" * 80)
